# Proyecto P05: Dashboard de COVID-19 — Guía del ejercicio

En este proyecto vas a construir un dashboard comparativo de la pandemia de COVID-19 por país, desde los archivos públicos de la Universidad Johns Hopkins hasta una app de **Streamlit** con cinco rankings, un mapa y una descarga en CSV. Integra lo practicado en la sección 8: preparación de datos con **pandas**, medidas con **DuckDB**, gráficas con **Plotly** y diseño de dashboards.

- La app que debes construir se llama `p05-dashboard-covid-app.py` y va en esta misma carpeta.
- Cuando termines, compara tu trabajo con la solución: [p05-dashboard-covid-solucion.ipynb](p05-dashboard-covid-solucion.ipynb) y [p05-dashboard-covid-app.py](p05-dashboard-covid-app.py).

## Contexto

Entre 2020 y 2023, el Centro de Ciencia e Ingeniería de Sistemas (CSSE) de la Universidad Johns Hopkins publicó a diario los casos confirmados, las muertes y las recuperaciones de COVID-19 de cada país. Esas series se convirtieron en la referencia de periodistas, gobiernos e investigadores.

Una organización de salud pública quiere un tablero de una sola página que responda cinco preguntas: qué países acumularon más contagios, cuáles más muertes, dónde fue mayor y menor la tasa de mortalidad, y dónde fue mayor la tasa de recuperación. Además, quiere poder elegir la fecha de corte, decidir cuántos casos necesita un país para entrar en los rankings de tasas y descargar la tabla con la que se hizo cada ranking.

## Datos

Los tres archivos están en la carpeta [`datos/`](datos/) y son una copia de las series de tiempo globales del CSSE:

| Archivo | Original | Contenido |
| --- | --- | --- |
| `casos-confirmados.csv` | `time_series_covid19_confirmed_global.csv` | Casos confirmados acumulados |
| `muertes.csv` | `time_series_covid19_deaths_global.csv` | Muertes acumuladas |
| `recuperados.csv` | `time_series_covid19_recovered_global.csv` | Recuperados acumulados |

Fuente: [COVID-19 Data Repository](https://github.com/CSSEGISandData/COVID-19) del Centro de Ciencia e Ingeniería de Sistemas (CSSE) de la Universidad Johns Hopkins, series de tiempo globales (`csse_covid_19_data/csse_covid_19_time_series/`). El repositorio se archivó en marzo de 2023 y sus datos se publican con licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Cita la fuente en tu dashboard.

Los tres archivos tienen el mismo formato **ancho**, con una fila por país o provincia:

| Columna | Descripción |
| --- | --- |
| `Province/State` | Provincia o territorio; vacía cuando el país se reporta completo |
| `Country/Region` | País o región |
| `Lat`, `Long` | Coordenadas aproximadas |
| `1/22/20` … `3/9/23` | Una columna por día (1143 en total), con la fecha en formato `M/D/YY` |

Particularidades:

- Los valores son **acumulados**: cada columna contiene el total desde el inicio de la pandemia hasta ese día, no los casos nuevos del día.
- Algunos países (por ejemplo, Australia, Canadá, China, Francia o Reino Unido) se reparten en varias filas, una por provincia o territorio, y no siempre con las mismas filas en los tres archivos.
- Confirmados y muertes tienen 289 filas; recuperados, 274.
- Hay entradas de `Country/Region` que no son países, como cruceros y eventos deportivos.

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta (`uv run jupyter lab` desde la raíz del repositorio), y lee los archivos con rutas relativas (`datos/...`).
- Usa pandas para preparar los datos y DuckDB para comprobar las medidas con SQL; las gráficas del notebook, con Plotly.
- Escribe la app en `p05-dashboard-covid-app.py` y lánzala desde la raíz del repositorio:

  ```bash
  uv run streamlit run proyectos/p05-dashboard-covid/p05-dashboard-covid-app.py
  ```

- Después de cada tarea, escribe en la celda de respuesta lo que encontraste: las cifras que lo prueban y las decisiones que tomaste.

*El enunciado original del curso no se conserva; estas instrucciones se reconstruyeron a partir de la solución, que se hizo en una herramienta de BI.*

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "png"
pio.defaults.default_width = 900
pio.defaults.default_height = 450

DATOS = Path("datos")
ARCHIVOS = {"confirmados": "casos-confirmados.csv", "muertes": "muertes.csv", "recuperados": "recuperados.csv"}
DESCRIPTIVAS = ["Province/State", "Country/Region", "Lat", "Long"]

## Tarea 1. Importar las series

1. Lee los tres archivos en un diccionario de DataFrames, uno por medida.
2. Revisa en cada uno el número de filas, el número de columnas de fecha, la primera y la última fecha y el número de países o regiones.
3. Compara las filas (pares provincia–país) de los tres archivos. ¿Tienen confirmados y muertes las mismas filas? ¿Qué filas de confirmados no aparecen en recuperados, y al revés?

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 2. Pasar del formato ancho al largo

1. Escribe una función que pase un archivo de formato ancho a largo con `melt`, dejando fijas las cuatro columnas descriptivas: una fila por provincia y día, con las columnas `fecha` y el valor de la medida.
2. Convierte `fecha` con `pd.to_datetime(..., format="%m/%d/%y")`, para que `1/22/20` se lea como 22 de enero de 2020 sin depender de la configuración regional.
3. Comprueba que no quedan valores nulos ni fechas mal leídas.
4. Rellena `Province/State` vacía con `""`. Comprueba qué pasa si no lo haces y agrupas por provincia y país con `groupby`.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 3. Combinar en una sola tabla

1. Intenta primero unir las tres series en formato largo por `Province/State`, `Country/Region` y `fecha` con combinaciones izquierdas. Cuenta cuántas filas quedan sin recuperados y revisa qué le pasa a Canadá.
2. Revisa también si las coordenadas `Lat`/`Long` coinciden entre confirmados y recuperados. ¿Conviene usarlas como clave de la unión?
3. Por lo anterior, **agrega a nivel país antes de unir**: para cada serie, calcula el valor diario de cada país como la suma, por provincia, del máximo acumulado hasta ese día (`cummax` por provincia y después suma por país y fecha). Luego une las tres series por país y fecha (`validate="one_to_one"`).
4. Comprueba que la tabla `covid` tiene una fila por país y día, sin valores nulos.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 4. Medidas por país

1. Escribe una función de pandas que calcule el total de un país a una fecha de corte: el **máximo** de cada provincia hasta esa fecha y la **suma** de esos máximos por país. Como la serie es acumulada, no sumes todos los días.
2. Escribe la misma medida en **DuckDB SQL** con dos niveles: una subconsulta o CTE con `MAX` por provincia y una consulta exterior con `SUM` por país. Pasa la fecha de corte como parámetro.
3. Comprueba con `pd.testing` que pandas, DuckDB y la tabla `covid` de la Tarea 3 dan exactamente lo mismo para confirmados, muertes y recuperados.
4. Calcula las tasas con **división segura** (si un país no tiene casos, la tasa queda vacía en lugar de dar error o infinito):

   $$\text{Mortalidad} = \frac{\text{Muertes}}{\text{Confirmados}} \qquad \text{Recuperación} = \frac{\text{Recuperados}}{\text{Confirmados}}$$

   En pandas, divide entre `confirmados.where(confirmados > 0)`; en SQL, entre `NULLIF(confirmados, 0)`.
5. Reúne todo en una función `tabla_por_pais(corte)` que devuelva una fila por país con confirmados, muertes, recuperados y las dos tasas en porcentaje.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 5. Revisar los problemas de los datos

Antes de ordenar países, comprueba con los datos cada uno de estos problemas y decide cómo tratarlo:

1. **Recuperados.** Grafica el total mundial diario de confirmados y de recuperados. ¿Hasta qué fecha se actualizó la serie de recuperados y qué valores tiene después? ¿Qué le pasa a la tasa de recuperación si divides los recuperados entre los confirmados de 2023? Busca también países cuya serie de recuperados dejó de crecer mucho antes.
2. **Países sin recuperados.** ¿Qué países nunca informaron recuperados? ¿Deberían aparecer en un ranking de recuperación con tasa 0?
3. **Entradas que no son países.** Localiza las entradas de `Country/Region` que no son países y las filas de provincia sin coordenadas. Decide cuáles excluyes de los rankings y cuáles se quedan dentro del total de su país.
4. **Pocos casos.** Ordena los países por mortalidad sin ningún filtro y revisa los primeros. Elige un mínimo de casos para los rankings de tasas y justifícalo.
5. **Acumulados que bajan.** Cuenta cuántas veces baja de un día a otro alguna serie acumulada y compara el máximo de cada provincia con su último valor. ¿Cuánto cambian los totales por usar el máximo?

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 6. Rankings

Escribe una función `rankings(tabla, n, minimo)` que devuelva las cinco tablas del dashboard, cada una con el país y la medida, ordenada para responder su pregunta:

1. "Países con más contagios": confirmados, de mayor a menor.
2. "Países con más decesos": muertes, de mayor a menor.
3. "Países con mayor tasa de mortalidad": mortalidad, de mayor a menor.
4. "Países con menor tasa de mortalidad": mortalidad, de menor a mayor.
5. "Países con mayor índice de recuperación": recuperación, de mayor a menor.

Aplica las decisiones de la Tarea 5 (entradas que no son países, mínimo de casos, países sin recuperados) **antes** de ordenar. Muestra las tasas con dos decimales y haz una gráfica de barras de uno de los rankings.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 7. Dashboard en Streamlit

Escribe `p05-dashboard-covid-app.py` con:

1. Carga de los tres CSV en caché (`@st.cache_data`), leyendo la carpeta con `Path(__file__).parent / "datos"` para que funcione al lanzar la app desde la raíz del repositorio.
2. Una barra lateral con la **fecha de corte**, el **mínimo de casos** para entrar en los rankings de tasas y el **número de países** por tabla (top N).
3. Las cinco tablas de rankings con `st.dataframe` y `st.column_config` (tasas con dos decimales), cada una con su título.
4. Una leyenda con tu nombre como autor y la cita de la fuente y su licencia.
5. Un mapa con `px.scatter_geo` a partir de `Lat` y `Long` (descarta las filas sin coordenadas).
6. Un `st.download_button` que descargue en CSV la tabla por país de la fecha de corte; sustituye a la exportación a PDF del tablero original.

Prueba la app con `streamlit.testing.v1.AppTest` desde este notebook: que se ejecute sin excepciones, muestre cinco tablas y que sus rankings coincidan con los de la Tarea 6.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] Las tres series están en formato largo, con `fecha` de tipo fecha, y combinadas a nivel país en una tabla sin nulos.
- [ ] Los totales por país no suman varias veces los valores acumulados (el total de un país no puede superar su población), y pandas y DuckDB dan lo mismo.
- [ ] Las tasas usan división segura y se muestran con dos decimales.
- [ ] La tasa de recuperación usa los confirmados a la misma fecha que los recuperados, sin países que nunca informaron recuperados.
- [ ] Los rankings de tasas exigen un mínimo de casos y excluyen las entradas que no son países.
- [ ] La app tiene las cinco tablas, el autor, la fuente, el mapa, los controles de la barra lateral y la descarga en CSV, y pasa la prueba con `AppTest`.

Cuando termines, compara tu trabajo con la solución: [p05-dashboard-covid-solucion.ipynb](p05-dashboard-covid-solucion.ipynb).